In [1]:
import numpy as np
import os
os.chdir('/orcd/data/omarabu/001/opitcho/assignment1-basics/')

In [5]:
# Loading npz
owt_tokens = np.load('data/owt_train_tokens.npz')
tinystories_tokens = np.load('data/ts_train_tokens.npz')

In [9]:
owt_tokens = owt_tokens['tokens']
tinystories_tokens = tinystories_tokens['tokens']

In [10]:
print(owt_tokens.shape)
print(tinystories_tokens.shape)

(2757135627,)
(554658421,)


In [ ]:
import torch

s = torch.tensor(0,dtype=torch.float32)
for i in range(1000):
    s += torch.tensor(0.01,dtype=torch.float32)
print(s)
s = torch.tensor(0,dtype=torch.float16)
for i in range(1000):
    s += torch.tensor(0.01,dtype=torch.float16)
print(s)
s = torch.tensor(0,dtype=torch.float32)
for i in range(1000):
    s += torch.tensor(0.01,dtype=torch.float16)
print(s)
s = torch.tensor(0,dtype=torch.float32)
for i in range(1000):
    x = torch.tensor(0.01,dtype=torch.float16)
    s += x.type(torch.float32)
print(s)

# BF16
s = torch.tensor(0,dtype=torch.bfloat16)
for i in range(1000):
    s += torch.tensor(0.01,dtype=torch.bfloat16)
print(f"Bfloat16: {s}")


tensor(10.0001)
tensor(9.9531, dtype=torch.float16)
tensor(10.0021)
tensor(10.0021)
Bfloat16: 4.0


In [ ]:
## Compare DDP vs Single GPU Checkpoints

In [2]:
import torch
from pathlib import Path

# Load checkpoints
checkpoint_dir = Path("checkpoints/naive_ddp")
ddp_ckpt = torch.load(checkpoint_dir / "final_ddp.pt", map_location="cpu")
single_gpu_ckpt = torch.load(checkpoint_dir / "final_single_gpu.pt", map_location="cpu")

print(f"DDP final loss: {ddp_ckpt['loss']:.6f}")
print(f"Single GPU final loss: {single_gpu_ckpt['loss']:.6f}")

DDP final loss: 1.649542
Single GPU final loss: 0.000672


In [3]:
# Compare model state dicts
ddp_state = ddp_ckpt["model_state_dict"]
single_state = single_gpu_ckpt["model_state_dict"]

print(f"Number of parameters: {len(ddp_state)}")

# Check if all keys match
assert ddp_state.keys() == single_state.keys(), "Keys mismatch!"

# Compare each parameter
all_match = True
max_diff = 0.0
for key in ddp_state.keys():
    ddp_param = ddp_state[key]
    single_param = single_state[key]
    
    if not torch.allclose(ddp_param, single_param, atol=1e-6):
        diff = (ddp_param - single_param).abs().max().item()
        max_diff = max(max_diff, diff)
        print(f"Mismatch in {key}: max diff = {diff:.2e}")
        all_match = False

if all_match:
    print("All parameters match exactly!")
else:
    print(f"\nMax difference across all params: {max_diff:.2e}")

Number of parameters: 436
Mismatch in word_embedding.weights: max diff = 1.63e-03
Mismatch in network.0.norm1.gamma: max diff = 9.64e-04
Mismatch in network.0.norm2.gamma: max diff = 1.11e-03
Mismatch in network.0.mha.QKV_proj.weights: max diff = 1.51e-03
Mismatch in network.0.mha.O_proj.weights: max diff = 1.38e-03
Mismatch in network.0.ffn.w1.weights: max diff = 1.51e-03
Mismatch in network.0.ffn.w2.weights: max diff = 1.66e-03
Mismatch in network.0.ffn.w3.weights: max diff = 1.55e-03
Mismatch in network.1.norm1.gamma: max diff = 9.35e-04
Mismatch in network.1.norm2.gamma: max diff = 1.24e-03
Mismatch in network.1.mha.QKV_proj.weights: max diff = 1.64e-03
Mismatch in network.1.mha.O_proj.weights: max diff = 1.27e-03
Mismatch in network.1.ffn.w1.weights: max diff = 1.69e-03
Mismatch in network.1.ffn.w2.weights: max diff = 1.65e-03
Mismatch in network.1.ffn.w3.weights: max diff = 1.61e-03
Mismatch in network.2.norm1.gamma: max diff = 9.91e-04
Mismatch in network.2.norm2.gamma: max diff

In [4]:
# Summary statistics for weight differences
diffs = []
for key in ddp_state.keys():
    diff = (ddp_state[key].float() - single_state[key].float()).abs()
    diffs.append(diff.mean().item())

import numpy as np
diffs = np.array(diffs)
print(f"Mean absolute difference: {diffs.mean():.2e}")
print(f"Max absolute difference:  {diffs.max():.2e}")
print(f"Min absolute difference:  {diffs.min():.2e}")

Mean absolute difference: 2.81e-04
Max absolute difference:  1.37e-03
Min absolute difference:  0.00e+00


In [1]:
import torch
import torch.distributed as dist

In [ ]:
x = torch.ones(10)
handle = dist.all_reduce(x, op=dist.ReduceOp.SUM, async_op=True)
print(x)
handle.synchronize


In [9]:
world_size = 2
mp.spawn(Worker.worker, args=(world_size,), nprocs=world_size, join=True)

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/home/opitcho/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/opitcho/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python3.12/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'Worker.worker' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/home/opitcho/.local/share/uv/python/cpython-3.12.11-linux-x86_64-gnu/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/opitcho/.local/share/uv/python

ProcessExitedException: process 1 terminated with exit code 1